[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/07-integrations/01-fastapi_match_service.ipynb)

In [1]:
# !pip install mbox fastapi uvicorn httpx

# Exposing M|BOX as a FastAPI Service

`06-agentic-ai/03` and `04` exposed an M|BOX index as an MCP tool, for LLM and agent clients specifically. Most systems that need a fuzzy lookup aren't an LLM at all, they're another backend service, a mobile app, a frontend, something in a completely different language that just wants to `POST` a query and get a score back. That's what a plain HTTP service is for.

In this notebook you will:

1. Build an M|BOX index once and save it, the same reusable pattern as `06-agentic-ai/04`
2. Write a small FastAPI service that loads that index once at startup and exposes it as `/match`
3. Launch it as a real subprocess and call it with real HTTP requests, not an in-process import
4. See what Pydantic validation catches for free, before a bad request ever reaches M|BOX
5. Shut the service down cleanly

> Note: this notebook launches a real `uvicorn` server as a background process on `localhost` and talks to it over HTTP. It stops the server in the last cell, run the notebook to the end.

In [1]:
import pandas as pd

## 1. Build the index once

Same reasoning as every server in this cookbook: an index is expensive enough to build, and cheap enough to reload, that a service should build it once, offline, and load the compiled result at startup, not reconstruct it from a CSV on every restart.

In [2]:
from mbox.indexing import TableIndexer

catalog = pd.read_csv("datasets/product_catalog.csv")
index = TableIndexer.create_index(catalog, index_columns=["product_name"], tmp_dir="tmp_index_build")
index.to_binary("product_catalog_index.zip")
print("Saved product_catalog_index.zip")

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object
Saved product_catalog_index.zip


## 2. The service

A small FastAPI app: one `/health` endpoint, one `/match` endpoint. The index loads once, in a `lifespan` handler that runs at process startup, not per-request, and every request thereafter reuses the same loaded `TableIndex` object.

In [3]:
%%writefile match_service.py
"""A FastAPI service exposing an M|BOX-backed product search over HTTP."""
from contextlib import asynccontextmanager
from typing import List

import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

from mbox.indexing import TableIndex
from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

state = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    state["index"] = TableIndex.load_binary("product_catalog_index.zip")
    state["catalog"] = pd.read_csv("datasets/product_catalog.csv")
    yield
    state.clear()


app = FastAPI(title="M|BOX Match Service", lifespan=lifespan)


class MatchRequest(BaseModel):
    query: str
    max_results: int = 3


class MatchCandidate(BaseModel):
    product_name: str
    score: int


class MatchResponse(BaseModel):
    found: bool
    matches: List[MatchCandidate]


@app.get("/health")
def health():
    return {"status": "ok", "indexed_rows": len(state["catalog"])}


@app.post("/match", response_model=MatchResponse)
def match(request: MatchRequest):
    config = TableRecallConfig(
        fields=[TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                        minimum_quality=0, weight=100, mode=TableRecallMode.APPROX)],
        max_results=request.max_results, min_total_match_value=0, include_field_scores=True
    )
    result = state["index"].match(queries=pd.DataFrame({"product_name": [request.query]}), config=config)

    if len(result) == 0 or result["index_row"].iloc[0] == -1:
        return MatchResponse(found=False, matches=[])

    matches = [
        MatchCandidate(product_name=row["product_name_candidate"], score=int(row["product_name_score"]))
        for _, row in result.iterrows()
    ]
    return MatchResponse(found=True, matches=matches)


Overwriting match_service.py


## 3. Launch it as a real process

A real client doesn't import this file, it starts the process and talks to it over the network. `subprocess.Popen` launches `uvicorn` exactly the way a deployment would, then a short polling loop waits for `/health` to answer before sending real traffic.

In [4]:
import subprocess
import sys
import time
import httpx

process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "match_service:app", "--port", "8123"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

base_url = "http://127.0.0.1:8123"
ready = False
for _ in range(30):
    try:
        if httpx.get(f"{base_url}/health", timeout=1).status_code == 200:
            ready = True
            break
    except httpx.TransportError:
        pass
    time.sleep(0.5)

print("Service ready:", ready)
httpx.get(f"{base_url}/health").json()

Service ready: True


{'status': 'ok', 'indexed_rows': 4}

## 4. Call it like any other HTTP client would

No M|BOX import in this cell, no Python-specific object, just JSON over HTTP.

In [5]:
response = httpx.post(f"{base_url}/match", json={"query": "extendd batery pak", "max_results": 3})
print(response.status_code)
response.json()

200


{'found': True,
 'matches': [{'product_name': 'Extended Battery Pack', 'score': 60}]}

A typo'd query, resolved over the network, back as plain JSON any client can parse. A query that matches nothing comes back the same shape, `found: false`, not an error, an empty result is a normal, well-formed answer.

In [6]:
response = httpx.post(f"{base_url}/match", json={"query": "bluetooth headphones"})
print(response.status_code)
response.json()

200


{'found': False, 'matches': []}

## 5. What Pydantic catches before M|BOX ever runs

`MatchRequest` declares `query` as required. Send a request without it.

In [7]:
response = httpx.post(f"{base_url}/match", json={"max_results": 3})
print(response.status_code)
response.json()

422


{'detail': [{'type': 'missing',
   'loc': ['body', 'query'],
   'msg': 'Field required',
   'input': {'max_results': 3}}]}

A `422`, with a precise message about exactly which field is missing, and M|BOX never even ran. That validation is not something this notebook wrote, it comes free from declaring `MatchRequest` as a Pydantic model, the same schema-from-types idea `06-agentic-ai/03` highlighted for MCP applies here too, just enforced at the HTTP boundary instead of a tool schema. FastAPI also generates interactive API docs from these same models automatically, visit `/docs` on a running instance in a browser to see the schema and try requests by hand.

## 6. Shut it down

Always stop what you started. Leaving a `uvicorn` process running in the background after the notebook is done is the kind of thing that quietly holds a port, or a file lock on the index, until something else fails for a reason that isn't obvious.

In [8]:
process.terminate()
process.wait(timeout=5)
print("Service stopped.")

Service stopped.


## 7. Practical notes

**Load the index in `lifespan`, not at module import time.** A `lifespan` handler runs exactly once, when the ASGI server starts the app, which is the right place for a one-time, possibly-slow load. Loading at bare module level works too for a simple case like this one, but `lifespan` is where cleanup, closing connections, releasing resources, belongs, and it keeps import-time side effects out of the module itself.

**Each worker process loads its own copy of the index.** Running `uvicorn` with multiple workers, or behind a process manager that starts several instances, means each one independently calls `load_binary()` once at its own startup. That's the correct behavior, not a bug to route around, the index is read-only after loading, so there's nothing to synchronize between workers.

**Validation errors are not M|BOX errors.** A `422` here means the request was malformed before M|BOX ever saw it. Don't conflate that with `found: false`, which is M|BOX's own, well-formed answer that nothing matched. A client needs to tell those two apart, one means "fix your request," the other means "the request was fine, there was just no good match."

**This is the general-purpose integration surface.** `06-agentic-ai/03` and `04`'s MCP servers exist because MCP clients, agents, expect MCP specifically. Every other kind of caller, a web frontend, a mobile app, a service written in a different language entirely, just wants JSON over HTTP, and that's what this notebook builds.

## Next steps

- **`02-sql_database_integration.ipynb`** - load the data this service indexes from a real database instead of a CSV
- **`03-pandas_etl_pipeline.ipynb`** - run this same kind of resolution as a stage inside a larger data pipeline